**Research Paper Question Answering Bot Using RAG**


**Aim:** The aim of this project is build to Research Paper Question Answer Bot using the Retrieval-Augmented Generation (RAG) technique. the system answer user questions based only on uploaded research paper instead of relying on the language model's general knowledge.

**Step 1:** Install Required Libraries



*  Installing all the libraries needed for this project.


In [1]:
!pip -q install \
langchain==1.3.14 \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain_huggingface \
langchain_groq \
langchain-classic \
chromadb \
pypdf \
sentence-transformers

**Step 2:** Upload Research Paper


*  Uploading the research papers from my laptop. These PDFs are what the bot will use to answer questions.


In [2]:
from google.colab import files
upload=files.upload()

Saving REALM.pdf to REALM.pdf
Saving Dense Passage Retrieval for Open-Domain Question Answering.pdf to Dense Passage Retrieval for Open-Domain Question Answering.pdf
Saving Sentence Embeddings using Siamese BERT-Networks.pdf to Sentence Embeddings using Siamese BERT-Networks.pdf
Saving RoBERTa.pdf to RoBERTa.pdf
Saving Language Models are Few-Shot Learners (GPT-3).pdf to Language Models are Few-Shot Learners (GPT-3).pdf
Saving Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf to Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Saving BERT.pdf to BERT.pdf
Saving Attention Is All You Need.pdf to Attention Is All You Need.pdf


**Step 3:** Load the research paper


*   Loading the uploaded PDFs using PyPDFLoader so I can read the text from them.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
documents=[]
for file_name in upload.keys():
  loader=PyPDFLoader(file_name)
  pages=loader.load()
  documents.extend(pages)
print("Number of PDFs uploaded:", len(upload))
print("total pages:" ,len(documents))

/tmp/ipykernel_2176/4057452513.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of PDFs uploaded: 8
total pages: 174


**Step 4:** Split the Documents into Chunks


* Large Language Models cannot efficiently process very large documents.

Therefore, the document is divided into smaller overlapping chunks using RecursiveCharacterTextSplitter.  



In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size= 1000,
    chunk_overlap=200
)
chunks=text_splitter.split_documents(documents)
print("total chunks:", len(chunks))

print("\nSample Chunk Content:")
print(chunks[0].page_content)
print("\nSample Chunk Metadata:")
print(chunks[0].metadata)

total chunks: 807

Sample Chunk Content:
arXiv:2002.08909v1  [cs.CL]  10 Feb 2020
REALM: Retrieval-Augmented Language Model Pre-Training
Kelvin Guu * 1 Kenton Lee * 1 Zora T ung1 Panupong Pasupat 1 Ming-Wei Chang 1
Abstract
Language model pre-training has been shown to
capture a surprising amount of world knowledge,
crucial for NLP tasks such as question answer-
ing. However, this knowledge is stored implic-
itly in the parameters of a neural network, requir-
ing ever-larger networks to cover more facts. To
capture knowledge in a more modular and inter-
pretable way, we augment language model pre-
training with a latent knowledge retriever, which
allows the model to retrieve and attend over doc-
uments from a large corpus such as Wikipedia,
used during pre-training, ﬁne-tuning and infer-
ence. For the ﬁrst time, we show how to pre-
train such a knowledge retriever in an unsuper-
vised manner, using masked language model-
ing as the learning signal and backpropagating
through a retrieva

**Step 5:** Generate First Embedding


*   Each text chunk is converted into a numerical vector using the all-MiniLM-L6-v2 embedding model.

**Model Used :** sentence-transformers/all-MiniLM-L6-v2



In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


**step 6:** Create the Vector Database


*   Storing all these vectors in a Chroma database so I can search through them later.


In [6]:
from langchain_community.vectorstores import Chroma
vector_db =Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)
print("total vectors:", vector_db._collection.count())

total vectors: 807


**Step 7:** Create the retriever


*  Creating a retriever that will find the top 3 most relevant chunks for any question I ask.



In [7]:
retriever=vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
print("retreiver created successfully")

retreiver created successfully


**Step 8:** Connect the Large Language Model



*   I connected Groq API key is securely loaded from Google Colab secrets.

Model = llama-3.1-8b-instant



In [8]:
!pip -q install langchain-groq

In [9]:
from google.colab import userdata
groq_API_key = userdata.get("groq_API_key")
print("API Key loaded successfully.")

API Key loaded successfully.


In [10]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key=groq_API_key,
    model_name="llama-3.1-8b-instant",
    temperature=0
)
print("Groq LLM connected successfully.")

Groq LLM connected successfully.


**Step 9:** Create the Prompt Template


*   A prompt template controls how the language model responds.



In [11]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
You are a helpful research paper assistant.

Use only the given context to answer the question.
write the answer in simple english.
do not include labels as "obtion 1", "obtion 2", or bullet numbers
If the answer is not found in the context, reply with "I don't know."

Context:
{context}

Question:
{question}

Answer:

""")

print("Prompt created seccessfully.")

Prompt created seccessfully.


**Step 10:** Generate Answer using the RAG Pipeline


*   This step executes the complete RAG pipeline.



In [12]:
question = input("Enter your question: ")
docs = retriever.invoke(question)

context="\n\n".join([doc.page_content for doc in docs])

messages=prompt.invoke({
    "context": context,
    "question": question
})
response = llm.invoke(messages)
answer =response.content.strip()

print("\n"+"=" * 60)
print("Answer")
print("="* 60)
print(answer)

if answer.lower() in ["i don't know.","i don't know"]:
    print("\n" +"="*60)
    print("Top 3 Supporting Sources")
    print("="*60)
    print("No relevant supporting sources found.")
else:
    print("\n"+"="*60)
    print("Top 3 Supporting Sources")
    print("=" * 60)

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source","Unknown")
        page = doc.metadata.get("page","Unknown")

        print(f"\nSource {i}")
        print(f"Paper: {source}")

        if isinstance(page, int):
            print(f"Page : {page + 1}")
        else:
            print(f"Page : {page}")

        print("\nSupporting Passage:")
        passage = doc.page_content

        if "Figure" in passage:
            passage = passage[passage.find("Figure"):]

        print(passage[:400])
        print("-"*60)

Enter your question: what is RAG?

Answer
RAG is a type of language model that is strongly grounded in real factual knowledge, such as Wikipedia. It is designed to be more factual and less likely to "hallucinate" than previous language models.

Top 3 Supporting Sources

Source 1
Paper: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page : 17

Supporting Passage:
blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found
at https://huggingface.co/rag/
2https://github.com/pytorch/fairseq
3https://github.com/huggingface/transformers
17
------------------------------------------------------------

Source 2
Paper: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page : 10

Supporting Passage:
Broader Impact
This work offers several positive societal beneﬁts over previous work: the fact that it is more
strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less
with generations that are

**step 11:** Load the Second Embedding Model


*   This step loads the second embedding model BAAI/bge-small-en-v1.5. It is used to compare its retrieval performance with the first embedding model (all-MiniLM-L6-v2).



**Note :** I didn't use OpenAI embeddings here because it needs a paid key. Instead I compared tow open-source models all-MiniLM-L6_v2 and BAAI/bge-small-en-v1.5 to see the difference between a smaller/faster models and a slightly bigger/better model.

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model_2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
print("Second embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Second embedding model loaded successfully.


In [14]:
from langchain_community.vectorstores import Chroma
vector_db_2 = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model_2
)
print("total vectors:", vector_db_2._collection.count())

total vectors: 1614


In [15]:
retriever_2 = vector_db_2.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
print("Second retriever created successfully.")

Second retriever created successfully.


In [16]:
question = input("Enter your question: ")

docs = retriever_2.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

messages = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(messages)

answer = response.content.strip()

print("\n" + "=" * 60)
print("Answer")
print("=" * 60)
print(response.content)

print("\n" + "=" * 60)
print("Top 3 Supporting Sources")
print("=" * 60)

if answer.lower() in ["i don't know", "i don't know."]:
    print("No relevant supporting sources found.")

else:

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")

        print(f"\nSource {i}")
        print(f"Paper : {source}")

        if isinstance(page, int):
            print(f"Page : {page + 1}")
        else:
            print(f"Page : {page}")

        print("\nSupporting Passage:")
        passage = " ".join(doc.page_content.split())


        if "Figure" in passage:
            passage = passage[passage.find("Figure"):]


        for word in [
            "Trm", "Lstm", "T1", "T2", "TN",
            "E1", "E2", "EN", "..."
        ]:
            passage = passage.replace(word, "")

        print(passage[:350])
        print("-" * 60)

Enter your question: what is RAG?

Answer
RAG is a model that uses the input sequence to retrieve text documents and use them as additional context when generating the target sequence.

Top 3 Supporting Sources

Source 1
Paper : Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page : 17

Supporting Passage:
blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found at https://huggingface.co/rag/ 2https://github.com/pytorch/fairseq 3https://github.com/huggingface/transformers 17
------------------------------------------------------------

Source 2
Paper : Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page : 2

Supporting Passage:
Figure 1, our models leverage two components: (i) a retriever pη(z|x) with parametersη that returns (top-K truncated) distributions over text passages given a queryx and (ii) a generatorpθ(yi|x,z,y 1:i−1) parametrized 1Code to run experiments with RAG has been open-sourced as part of the Hu

**Compare Two Embedding Models**

| Feature | all-MiniLM-L6-v2 | BAAI/bge-small-en-v1.5 |
|---------|------------------|------------------------|
| Speed | Faster | Slightly slower |
| Answer detail | Short, 1 point | More detailed, covers multiple points |

**Observation:**

I tested both embedding models using the question "What is RAG?". all-MiniLM-L6-v2 gave a general answer, while BAAI/bge-small-en-v1.5 produced a more accurate and relevant explanation. So, I chose BAAI/bge-small-en-v1.5 because it retrieved better context, even though it was slightly slower.

**Step 12:** create the MMR Retriever


*  Trying MMR retrieval now — instead of picking the top 3 most similar chunks (which can repeat the same info), MMR tries to pick chunks that are relevant but also a bit different from each other.




In [17]:
retriever_mmr = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 20,
        "lambda_mult": 0.8
    }
)
print("MMR retriever created successfully.")

MMR retriever created successfully.


In [18]:
question = input("Enter your question: ")

docs = retriever_mmr.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

messages = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(messages)
answer = response.content.strip()

print("\n" + "=" * 60)
print("Answer")
print("=" * 60)
print(response.content)

print("\n" + "=" * 60)
print("Top 3 Supporting Sources")
print("=" * 60)

if answer.lower() in ["i don't know", "i don't know."]:
    print("No relevant supporting sources found.")

else:

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")

        print(f"\nSource {i}")
        print(f"Paper: {source}")

        if isinstance(page, int):
            print(f"Page: {page + 1}")
        else:
            print(f"Page: {page}")

        passage = " ".join(doc.page_content.split())

        if "Figure" in passage:
            passage = passage[passage.find("Figure"):]


        for word in [
            "Trm", "Lstm", "T1", "T2", "TN",
            "E1", "E2", "EN", "..."
        ]:
            passage = passage.replace(word, "")

        print(passage[:350])
        print("-" * 60)

Enter your question: What is RAG?

Answer
RAG is a type of language model that is strongly grounded in real factual knowledge, such as Wikipedia. It is designed to be more factual and less likely to "hallucinate" than previous models.

Top 3 Supporting Sources

Source 1
Paper: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page: 17
blob/master/examples/rag/README.md and an interactive demo of a RAG model can be found at https://huggingface.co/rag/ 2https://github.com/pytorch/fairseq 3https://github.com/huggingface/transformers 17
------------------------------------------------------------

Source 2
Paper: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf
Page: 10
Broader Impact This work offers several positive societal beneﬁts over previous work: the fact that it is more strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less with generations that are more factual, and offers more control and interpreta

**Step 13:** Compare Retrieval Strategies


*   Here, I am not changing the embedding model.

Instead, I change only the retrieval strategy.



**1. Similarity Search**

Similarity Search returns the document chunks with the highest similarity scores.

Advantages:

*  Fast
*  Simple to use
*  Works well for direct factual questions



**2. MMR**
MMR Balances:


*  Relevance
*  Diversity

Instead of retrieving three similar chunks, MMR retrieves relevant chunks while reducing redundancy.



**Final Choice:**
I tested both Similarity Search and MMR using the question "What is RAG?". Both produced the same answer, but Similarity Search retrieved more directly relevant chunks. So, I chose Similarity Search for my final pipeline because it works well for factual questions.

**Step 14:** Test the RAG System

The Research Paper Answer Bot was tested using multiple questions from different research papers. The generated answer and supporting sources were verified to evaluate the retrievel quality and response accuracy.

In [19]:
test_questions =[
    "What is BERT?",
    "What is GPT-3?",
    "What is Transformer?",
    "What is RoBERTa?",
    "What is REALM?",
    "What is RAG?",
    "What is Attention?",
    "What is Encoder?",
    "What is Decoder?",
    "What is Sentence-BERT?"
]
for question in test_questions:
  print("\n"+"="*70)
  print("Question:", question)
  docs= retriever.invoke(question)
  context = "\n\n".join([doc.page_content for doc in docs])
  messages = prompt.invoke({
      "context": context,
      "question": question
  })
  response=llm.invoke(messages)
  answer = response.content.strip()
  print("\nAnswer:")
  print(response.content)
  print("\nTop 3 Supporting Sources:")

  if answer.lower() in ["i don't know", "i don't know."]:
        print("No relevant supporting sources found.")

  else:

      for i, doc in enumerate(docs, start=1):
        source=doc.metadata.get("source", "unknown")
        page=doc.metadata.get("page", "Unknown")
        if isinstance(page, int):
          page=page+1
        print(f"{i}. {source} (page {page})")


Question: What is BERT?

Answer:
BERT is a multi-layer bidirectional Transformer encoder.

Top 3 Supporting Sources:
1. RoBERTa.pdf (page 4)
2. BERT.pdf (page 3)
3. BERT.pdf (page 13)

Question: What is GPT-3?

Answer:
GPT-3 is a predecessor to another model, GPT-2.

Top 3 Supporting Sources:
1. Language Models are Few-Shot Learners (GPT-3).pdf (page 18)
2. Language Models are Few-Shot Learners (GPT-3).pdf (page 33)
3. Language Models are Few-Shot Learners (GPT-3).pdf (page 27)

Question: What is Transformer?

Answer:
The Transformer is a model architecture that uses multi-head attention in three different ways. It has an encoder and a decoder. The encoder and decoder are made up of stacked layers, each with two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network.

Top 3 Supporting Sources:
1. Attention Is All You Need.pdf (page 5)
2. Attention Is All You Need.pdf (page 3)
3. Language Models are Few-Shot Learners (GPT-3).p

**Testing Notes:**

8 out of 10 answers were correct. 2 had issues:
- GPT-3 answer was confusing (said GPT-3 came before GPT-2, which is wrong)
- Sentence-BERT gave "I don't know" because I didn't upload that paper

The bot correctly says "I don't know" when info is missing instead of
making things up, which is good for a RAG system.

**Step 15:** Stretch Goal - Streamlit User Interface

A simple Streamlit interface is created to make the Research Paper Question Answer Bot interactive. Users Can upload research papers, enter questions, and receive answer qenerated using the RAG pipeline.

In [24]:
%%writefile app.py
import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import tempfile
import os

st.set_page_config(page_title="Research Paper Bot", page_icon="📄")

with st.sidebar:
    st.title("Research Paper Bot")
    uploaded_file =st.file_uploader("Upload Research Paper", type="pdf", accept_multiple_files = True)
    groq_api_key=st.text_input("Enter Groq API Key", type="password")
    if st.button("Clear Chat"):
        st.session_state.chat_history=[]
        st.rerun()
st.title("Research Paper Question Answer Bot")
if "chat_history" not in st.session_state:
    st.session_state.chat_history=[]
if "rag_ready" not in st.session_state:
    st.session_state.rag_ready=False
if uploaded_file and groq_api_key and not st.session_state.rag_ready:
    documents = []
    uploaded_names = []

    for uploaded_file in uploaded_file:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
            temp_file.write(uploaded_file.read())
            pdf_path = temp_file.name

        loader = PyPDFLoader(pdf_path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = uploaded_file.name

        documents.extend(docs)
        uploaded_names.append(uploaded_file.name)

        os.remove(pdf_path)

    st.session_state.original_filenames = uploaded_names
    splitter =RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks=splitter.split_documents(documents)

    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_db=Chroma.from_documents(documents=chunks, embedding=embedding)
    st.session_state.retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

    st.session_state.llm =ChatGroq(
        groq_api_key=groq_api_key,
        model_name="llama-3.1-8b-instant",
        temperature=0
    )
    st.session_state.prompt = ChatPromptTemplate.from_template("""
You are a helpful research paper assistant.
Use only the given context to answer the question.
If the answer is not found in the context, reply with "I don't know."
Use the chat history to understand follow-up questions.

Chat History:
{chat_history}

Context:
{context}

Question:
{question}

Answer:
""")
    st.session_state.rag_ready = True
    st.success("File uploaded and indexed. You can ask questions now.")
for msg in st.session_state.chat_history:
    with st.chat_message("user"):
        st.write(msg["question"])
    with st.chat_message("assistant"):
        st.write(msg["answer"])
        if msg["sources"]:
            with st.expander("Supporting Sources"):
                for i, src in enumerate(msg["sources"], start=1):
                    st.write(f"{i}. {src['name']} (page {src['page']})")
if st.session_state.rag_ready:
    question=st.chat_input("Ask a question about the paper")

    if question:
        with st.chat_message("user"):
            st.write(question)

        docs =st.session_state.retriever.invoke(question)
        context="\n\n".join([doc.page_content for doc in docs])

        history_text="\n".join(
            [f"Q: {h['question']}\nA: {h['answer']}" for h in st.session_state.chat_history]
        )

        messages = st.session_state.prompt.invoke({
            "context": context,
            "question": question,
            "chat_history": history_text
        })

        with st.chat_message("assistant"):
            response = st.session_state.llm.invoke(messages)
            st.write(response.content)
            sources=[]
            if "i don't know" not in response.content.lower():
                with st.expander("Supporting Sources"):
                    for i, doc in enumerate(docs, start=1):
                        page = doc.metadata.get("page", 0)
                        page_display = page + 1 if isinstance(page, int) else page
                        source_name = doc.metadata.get("source", "Unknown")

                        st.write(f"{i}. {source_name} (page {page_display})")

                        sources.append({
                            "name": source_name,
                            "page": page_display
                        })
        st.session_state.chat_history.append({
            "question": question,
            "answer": response.content,
            "sources": sources
        })
else:
   st.success(
        f"{len(uploaded_file)} PDF(s) uploaded and indexed successfully. You can ask questions now."
)

Overwriting app.py


In [25]:
!pip -q install streamlit pyngrok

from pyngrok import ngrok
from google.colab import userdata
import subprocess, time

ngrok_token=userdata.get("ngrok_auth_token")
ngrok.set_auth_token(ngrok_token)

ngrok.kill()
!pkill -9 -f streamlit 2>/dev/null

streamlit_process=subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(5)

public_url=ngrok.connect(8501)
print("Your app is live at:", public_url)

^C
Your app is live at: NgrokTunnel: "https://latitude-kilometer-rebel.ngrok-free.dev" -> "http://localhost:8501"


**Challenges & Learnings:**

- Some retrieved chunks contained tables or less useful information instead of clear explanations.
- The bot answers only from the uploaded research papers. If the information is not available, it correctly replies "I don't know."
- Building the Streamlit interface and running it with ngrok in Google Colab required a few attempts.

**Conclusion:**

This project helped me understand the complete RAG pipeline, from loading and chunking PDFs to generating answers using retrieved context. I also learned how embedding models and retrieval strategies affect answer quality. Finally, I built a simple Streamlit chatbot to make the system interactive.